### 1.问题
如果我们希望 Agent 遵循特定领域的工作流：git约定、测试模式、代码审查清淡。如果我们把这些全塞进系统提示词中太浪费 -- 10个skill，每个2000个token，就是20000token，大部分可能跟当前任务毫无关系。

### 2.解决方案
1. 系统提示中放 Skill 名称（低成本）；
2. tool_result中按需放完整内容；
### 3. 工作原理
1. 每个 skill 是一个目录，包含SKILL.md文件和YAML frontmatter
```
skills/
  pdf/
    SKILL.md       # ---\n name: pdf\n description: Process PDF files\n ---\n ...
  code-review/
    SKILL.md       # ---\n name: code-review\n description: Review code\n ---\n ...
```
2. SkillLoader递归扫描SKILL.md文件，用目录名作为Skill标识。
```python
class SkillLoader:
    def __init__(self, skills_dir: Path):
        self.skills = {}
        for f in sorted(skills_dir.rglob("SKILL.md")):
            text = f.read_text()
            meta, body = self._parse_frontmatter(text)
            name = meta.get("name", f.parent.name)
            self.skill[name] = {"meta": meta, "body": body}

    def get_description(self) -> str:
        lines = []
        for name, skill in self.skills.items():
            desc = skill["meta"].get("description", "")
            lines.append(f" - {name}:{desc}")
        return "\n".join(lines)

    def get_content(self, name: str) -> str:
        skill = self.skill.get(name)
        if not skill:
            return f"Error: Unkonwn skill '{name}'"
        return f"<skill name=\"{name}\">\n{skill['body']}\n</skill>"
```
3. 第一层写入系统提示词。第二层不过是dispatch map中的又一个工具
```python
SYSTEM = f"""You are a coding agent at {WORKDIR}.
Skills available:
{SKILL_LOADER.get_descriptions()}"""

TOOL_HANDLERS = {
    # base tools
    "load_skill": lambda **kw: SKILL_LOADER.get_content(kw["name"])
}
```
模型一开始只需要知道有哪些SKill，需要时再加载完整内容。

In [ ]:
import os
import re
import subprocess
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

WORKDIR = Path.cwd()
client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"))

SKILLS_DIR = WORKDIR / "skills"

class SkillLoader:
    def __init__(self, skills_dir: Path):
        self.skills_dir = skills_dir
        self.skills = {}
        self._load_all()

    def _load_all(self):
        if not self.skills_dir.exists():
            return
        for f in sorted(self.skills_dir.rglob("SKILL.md")):
            text = f.read_text()
            meta, body = self._parse_frontmatter(text)
            name = meta.get("name", f.parent.name)
            self.skills[name] = {"meta": meta, "body": body, "path": str(f)}

    def _parse_frontmatter(self, text: str) -> tuple:
        """Parse YAML frontmatter between --- delimiters."""
        match = re.match()
